# 03 — Baselines y cinco familias
Resultados reales de CV sobre simulacion; sin evaluacion final.

In [1]:
from pathlib import Path
import sys, json
sys.dont_write_bytecode = True
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/sinteticos.py").is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from IPython.display import display, Image
from src.experimentos import seed_everything, holdout_indices, cv_indices, evaluate_cv, tune_grid


## Protocolo
Mismos tres folds temporales, features y preprocesamiento. ROC-AUC primaria; AP, F1, balanced accuracy y accuracy secundarias. Baseline prior y regla academica de importe relativo/estacionalidad. Familias: logistica regularizada, random forest (bagging), histogram boosting, MLP y SVM de kernel. No son cinco variantes de cuatro familias.

El intento fundacional TabPFN v2 se documenta abajo: instalado y un fold real terminado, pero CV completa limitada por recursos. LLM tiene justificacion tecnica sin ejecucion de prompting. SVM cubre una quinta familia convencional, no reemplaza ese requisito. Ver docs/metodologia_sintetica.md.

In [2]:
from src.ejecutar_estudio import run_candidates
results = run_candidates()
assert len(results) == 7, 'Revisar errores_candidatos.json; no hay cumplimiento con candidatos fallidos.'
display(results[['candidate','family','run_id','validation_roc_auc_mean','validation_roc_auc_std','gap_roc_auc_mean','fit_seconds_mean','warnings']])

C:\Users\PC\Documents\Codex\2026-11-10\pset-2-verificar-conexi-n-con\pset_3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/11/17 22:45:42 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at C:\Users\PC\Documents\Codex\2026-11-10\pset-2-verificar-conexi-n-con\pset_3\.venv\Lib\site-packages\mlflow\assistant\skills\mlflow-trace-instrumentation\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Reutilizando siete corridas reales verificadas del mismo dataset; sin nuevos entrenamientos.


,candidate,family,run_id,validation_roc_auc_mean,validation_roc_auc_std,gap_roc_auc_mean,fit_seconds_mean,warnings
0,dummy,baseline_ingenuo,bb2c1ad0c5804bf09ef3685b64d9cf62,0.500000,0.000000,0.000000,0.032122,0
1,regla_academica,baseline_regla,b3892c865b2243449630c33bbf50f485,0.518377,0.026910,0.003812,0.028571,0
2,logistica,lineal_regularizada,ff75495d50894332960411fa2e37fc8b,0.570464,0.002261,0.004606,0.039278,0
3,random_forest,bagging,2a5e517bf6f549cf853b7263091e7c04,0.555783,0.007920,0.121841,0.298438,0
4,hist_boosting,boosting,6a388c092fb542e192fdf4f42589a9ab,0.541535,0.014237,0.249435,0.217816,0
5,mlp,red_neuronal,e5a239d8dff94bb78f437ff94a119662,0.568070,0.007671,0.032468,0.237248,0
6,svm_rbf,kernel_svm,8e70234f2b544596a1e59578afb72e0c,0.554149,0.015608,0.103139,1.529683,0


In [3]:
print(json.dumps(json.loads((ROOT / 'docs/experimentos_resumen.json').read_text()), indent=2))

{
  "experiment": "pset3_mc_eventos_v1",
  "candidates_successful": 7,
  "failures": [],
  "exported_runs": 28,
  "holdout_evaluated": false,
  "git_sha": "UNAVAILABLE_PSET3_NOT_VERSIONED",
  "code_sha256": "55fb36817a2a412ab8c97d80967845ef7e2b8cc8fa6c0ef56b097265643bf3e7",
  "snowflake_status": "NOT_PUBLISHED_USER_SELECTED_LOCAL",
  "llm_status": "NOT_RUN_NO_LOCAL_WEIGHTS_OR_PROVIDER_CONFIGURED",
  "code_sha256_at_run_start": [
    "sha256:679356ec2be34e7b59360d894dda196a066a4bef1a351664b33783501acea6e5"
  ],
  "code_sha256_note": "End-of-run hash may include auxiliary files added while candidates ran; run tags record the start snapshot."
}


## Alcance
Los modelos se reajustan solo sobre desarrollo para registrar Pipeline, firma y ejemplo. No hay champion/challenger ni seleccion con hold-out. Resultados limitados al mundo simulado. Los warnings y fallos se conservan; ningun dato procede de Snowflake en esta fase local.

In [4]:
from src.fundacional import compare_foundation
foundation = compare_foundation()  # Recupera el intento real; no repite la carga pesada.
print(json.dumps(foundation, indent=2))

{
  "status": "BLOCKED_RESOURCE_BUDGET",
  "run_id": "b7c4e87d0d374625a2e1ab111f0821f1",
  "installed": "TabPFN10.2.0 Torch2.15.0",
  "weights_downloaded": "public V2 classifier, 29009539 bytes",
  "device": "cpu, 2 threads, n_estimators=1, cache and memory-saving enabled",
  "hardware": "GTX1650 4GiB; 19GiB RAM; torch CPU build",
  "observed_available_ram_gib_before_abort": 1.03,
  "elapsed_seconds_before_abort": 288.53892114257815,
  "completed_folds": 1,
  "cv_metrics_available": false,
  "holdout_used": false,
  "reason": "Resource budget reached while progressing through CPU folds: available RAM fell to about 1 GiB. Dedicated worker stopped. Partial fold results are retained, but no complete comparable 3-fold mean exists.",
  "license": "V2 weights Prior Labs Apache2+attribution; package Apache2. No login or license gate bypass.",
  "legacy_version_incompatibility": "tabpfn2.2.1 requires sklearn<1.7 and pandas<3; existing versions preserved.",
  "partial_folds": [
    {
      "run